In [1]:
!pip install datasets -q
!pip install git+https://github.com/amazon-science/chronos-forecasting.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requ

In [2]:
import sys
sys.path.insert(0, "/content/chronos-forecasting/src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from datasets import load_dataset
from chronos import ChronosPipeline
import torch
import warnings
warnings.filterwarnings('ignore')

2026-06-06 22:42:23.364475: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780785743.608965      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780785743.675551      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780785744.225460      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780785744.225500      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780785744.225503      58 computation_placer.cc:177] computation placer alr

In [3]:
def smape(actual, forecast):
    return 100 * np.mean(2 * np.abs(actual - forecast) / (np.abs(actual) + np.abs(forecast) + 1e-8))

def seasonal_naive(series, seasonality, prediction_length):
    if len(series) < seasonality:
        return np.full(prediction_length, series[-1] if len(series) > 0 else 0)
    last_season = series[-seasonality:]
    forecast = np.tile(last_season, (prediction_length // seasonality + 1))[:prediction_length]
    return forecast

def ets_forecast(series, prediction_length, seasonality):
    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        if len(series) > 2 * seasonality:
            model = ExponentialSmoothing(series, seasonal_periods=seasonality, trend=None, seasonal='add')
            fit = model.fit()
            return fit.forecast(prediction_length)
        else:
            return None
    except:
        return None

def arima_forecast(series, prediction_length):
    try:
        from statsmodels.tsa.arima.model import ARIMA
        if len(series) > 50:
            model = ARIMA(series, order=(5,1,0))
            fit = model.fit()
            return fit.forecast(prediction_length)
        else:
            return None
    except:
        return None

def theta_forecast(series, prediction_length, seasonality):
    try:
        from statsmodels.tsa.forecasting.theta import ThetaModel
        if len(series) > 2 * seasonality:
            model = ThetaModel(series, period=seasonality)
            fit = model.fit()
            return fit.forecast(prediction_length)
        else:
            return None
    except:
        return None

def evaluate_all_models(dataset_name, display_name, prediction_length, seasonality, max_series=None):
    print(f"Обработка: {display_name} ({dataset_name})")
    print(f"Горизонт: {prediction_length}, Сезонность: {seasonality}")
    
    try:
        dataset = load_dataset("autogluon/chronos_datasets", dataset_name, split="train")
        df = dataset.to_pandas()
        
        series_ids = df['id'].unique()
        n_series_total = len(series_ids)
        
        if max_series is None:
            n_series = n_series_total
            series_ids = series_ids
        else:
            n_series = min(max_series, n_series_total)
            series_ids = series_ids[:n_series]
        
        print(f"Всего рядов в датасете: {n_series_total}")
        print(f"Будет обработано: {n_series}")
        
        chronos_results = []
        snaive_results = []
        ets_results = []
        arima_results = []
        theta_results = []
        
        for idx, series_id in enumerate(series_ids):
            series_data = df[df['id'] == series_id]['target'].values[0]
            
            if not isinstance(series_data, np.ndarray):
                series_data = np.array(series_data)
            
            if len(series_data) < prediction_length + 50:
                continue
            
            train = series_data[:-prediction_length]
            test = series_data[-prediction_length:]
            
            try:
                forecast = pipeline.predict(
                    torch.tensor(train, dtype=torch.float32),
                    prediction_length=prediction_length,
                    num_samples=20
                )
                forecast_np = forecast.numpy()
                if forecast_np.ndim == 3:
                    forecast_np = forecast_np[0]
                forecast_chronos = np.median(forecast_np, axis=0)
                chronos_results.append({
                    'mae': mean_absolute_error(test, forecast_chronos),
                    'smape': smape(test, forecast_chronos)
                })
            except:
                pass
            
            forecast_snaive = seasonal_naive(train, seasonality, prediction_length)
            snaive_results.append({
                'mae': mean_absolute_error(test, forecast_snaive),
                'smape': smape(test, forecast_snaive)
            })
            
            # ETS
            forecast_ets = ets_forecast(train, prediction_length, seasonality)
            if forecast_ets is not None:
                ets_results.append({
                    'mae': mean_absolute_error(test, forecast_ets),
                    'smape': smape(test, forecast_ets)
                })
            
            # ARIMA
            forecast_arima = arima_forecast(train, prediction_length)
            if forecast_arima is not None:
                arima_results.append({
                    'mae': mean_absolute_error(test, forecast_arima),
                    'smape': smape(test, forecast_arima)
                })
            
            # Theta
            forecast_theta = theta_forecast(train, prediction_length, seasonality)
            if forecast_theta is not None:
                theta_results.append({
                    'mae': mean_absolute_error(test, forecast_theta),
                    'smape': smape(test, forecast_theta)
                })
            
            if (idx + 1) % 100 == 0:
                print(f"  Прогресс: {idx + 1}/{n_series}")
        
        result = {'dataset': display_name, 'horizon': prediction_length}
        
        if chronos_results:
            result['chronos_mae'] = np.mean([r['mae'] for r in chronos_results])
            result['chronos_smape'] = np.mean([r['smape'] for r in chronos_results])
            result['chronos_n'] = len(chronos_results)
        
        if snaive_results:
            result['snaive_mae'] = np.mean([r['mae'] for r in snaive_results])
            result['snaive_smape'] = np.mean([r['smape'] for r in snaive_results])
            result['snaive_n'] = len(snaive_results)
        
        if ets_results:
            result['ets_mae'] = np.mean([r['mae'] for r in ets_results])
            result['ets_smape'] = np.mean([r['smape'] for r in ets_results])
            result['ets_n'] = len(ets_results)
        
        if arima_results:
            result['arima_mae'] = np.mean([r['mae'] for r in arima_results])
            result['arima_smape'] = np.mean([r['smape'] for r in arima_results])
            result['arima_n'] = len(arima_results)
        
        if theta_results:
            result['theta_mae'] = np.mean([r['mae'] for r in theta_results])
            result['theta_smape'] = np.mean([r['smape'] for r in theta_results])
            result['theta_n'] = len(theta_results)
        
        print(f"\n  {display_name}: Chronos sMAPE={result.get('chronos_smape', 'N/A')}% (n={result.get('chronos_n', 0)})")
        
        return result
        
    except Exception as e:
        print(f"  Ошибка: {e}")
        return None

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.float32,
)

Device: cuda


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [5]:
datasets_config = [
    ("m4_yearly", "M4 Yearly", 6, 1),
    ("m4_quarterly", "M4 Quarterly", 8, 4),
    ("m4_monthly", "M4 Monthly", 18, 12),
    ("m4_weekly", "M4 Weekly", 13, 52),
    ("m4_daily", "M4 Daily", 14, 7),
    ("m4_hourly", "M4 Hourly", 48, 24),
    ("monash_electricity_hourly", "Electricity", 48, 24),
    ("monash_traffic", "Traffic", 48, 24),
    ("monash_weather", "Weather", 30, 12),
    ("monash_nn5_weekly", "NN5 Weekly", 8, 52),
]

In [6]:
all_results = []

for ds_name, disp_name, pred_len, season in datasets_config:
    result = evaluate_all_models(ds_name, disp_name, pred_len, season, max_series=500)
    if result:
        all_results.append(result)

Обработка: M4 Yearly (m4_yearly)
Горизонт: 6, Сезонность: 1


README.md: 0.00B [00:00, ?B/s]

m4_yearly/train-00000-of-00001.parquet:   0%|          | 0.00/5.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23000 [00:00<?, ? examples/s]

Всего рядов в датасете: 23000
Будет обработано: 500
  Прогресс: 100/500


...

monash_nn5_weekly/train-00000-of-00001.p(…):   0%|          | 0.00/64.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/111 [00:00<?, ? examples/s]

Всего рядов в датасете: 111
Будет обработано: 111


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


  Прогресс: 100/111

  NN5 Weekly: Chronos sMAPE=11.772082328796387% (n=111)


In [7]:
print("ИТОГОВАЯ ТАБЛИЦА (sMAPE, %)")

comparison_data = []
for r in all_results:
    row = {
        'Dataset': r['dataset'],
        'N_series': r.get('chronos_n', 0),
        'Chronos': round(r.get('chronos_smape', np.nan), 2),
        'Seasonal Naive': round(r.get('snaive_smape', np.nan), 2),
        'ETS': round(r.get('ets_smape', np.nan), 2),
        'ARIMA': round(r.get('arima_smape', np.nan), 2),
        'Theta': round(r.get('theta_smape', np.nan), 2),
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Chronos')
print(comparison_df.to_string(index=False))

comparison_df.to_csv("full_comparison_results.csv", index=False)

ИТОГОВАЯ ТАБЛИЦА (sMAPE, %)
     Dataset  N_series  Chronos  Seasonal Naive    ETS  ARIMA  Theta
    M4 Daily       500     2.70        3.240000   2.64   2.93   2.67
M4 Quarterly       379     6.31        7.870000   6.81   5.92   6.60
   M4 Weekly       359     6.31       14.520000   7.35   8.88   6.55
   M4 Hourly       414     7.83       13.910000  15.05  45.80  18.14
  M4 Monthly       500    10.74       13.780000  10.60  12.49  10.83
  NN5 Weekly       111    11.77       16.370001  12.06  12.18  12.09
   M4 Yearly       213    12.78       13.590000    NaN  12.96  12.66
 Electricity       321    14.02       14.020000  21.60  47.88  23.82
     Traffic       500    25.84       32.900000  62.09  85.87  50.37
     Weather       500   145.22       77.310000 164.58 106.26 164.45
